# Foundation Model Agents & the Pattern Catalogue

Notebook companion to the [Foundation Model Agents lesson](https://ml-viz.vercel.app/courses/agent-design-patterns/01-foundation-model-agents).

**What you'll build here:**
- Implement a minimal ReAct-style agent loop with a mocked LLM and tool registry
- Run a traced multi-step reasoning example and inspect each perception-reasoning-action cycle
- Demonstrate hallucination as a failure mode: FM without tools vs FM with tools
- Extend the agent with a new clock tool (your turn)

In [ ]:
import re
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#1a1d27',
    'axes.edgecolor':    '#2d3148',
    'axes.labelcolor':   '#e2e8f0',
    'xtick.color':       '#94a3b8',
    'ytick.color':       '#94a3b8',
    'text.color':        '#e2e8f0',
    'grid.color':        '#2d3148',
    'lines.linewidth':   1.5,
    'font.size':         11,
})
BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
YELLOW = '#fbbf24'
MUTED  = '#475569'

## The Perception-Reasoning-Action Loop

The canonical FM agent loop — popularised as **ReAct** (Reason + Act) — works like this:

```
while not done:
    observation = environment.observe()
    thought, action = FM.reason(goal, history, observation)
    result = tools.execute(action)
    history.append((thought, action, result))
    if action == 'finish': done = True
```

The FM sees: the original goal, the full history of (thought, action, result) tuples, and the current observation. It outputs a *thought* (free-form reasoning) and an *action* (structured tool call or 'finish').

Below we implement this with a **mocked LLM** — a function that returns hardcoded responses based on pattern matching. This lets us trace exactly what happens at each step without a real API call.

In [ ]:
# ── Tool registry ─────────────────────────────────────────────────────────────

def calculator(expression: str) -> str:
    """Safely evaluate a simple arithmetic expression."""
    try:
        # Only allow digits, operators, spaces, parentheses
        if not re.match(r'^[\d\s\.\+\-\*/\(\)]+$', expression):
            return "Error: unsafe expression"
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


def web_search(query: str) -> str:
    """Mocked web search — returns a hardcoded result for demo purposes."""
    if '15' in query and '7' in query:
        return "Search results: 15 × 7 = 105 (multiplication table)"
    return f"Search results for '{query}': No relevant results found."


TOOLS = {
    'calculator': calculator,
    'web_search':  web_search,
}

print("Available tools:", list(TOOLS.keys()))
print("calculator('15 * 7 + 3') →", calculator('15 * 7 + 3'))

In [ ]:
# ── Mocked LLM ────────────────────────────────────────────────────────────────

def mocked_llm(goal: str, history: list) -> dict:
    """
    Simulate an FM response. Returns {'thought': str, 'action': str, 'params': dict}.
    In a real system this calls an API (e.g. Anthropic / OpenAI).
    """
    step = len(history)

    if '15 * 7' in goal or '15 × 7' in goal:
        if step == 0:
            return {
                'thought': 'I need to compute 15 × 7 first, then add 3. I will use the calculator.',
                'action': 'calculator',
                'params': {'expression': '15 * 7'},
            }
        elif step == 1:
            prev_result = history[-1]['result']
            return {
                'thought': f'15 × 7 = {prev_result}. Now I add 3.',
                'action': 'calculator',
                'params': {'expression': f'{prev_result} + 3'},
            }
        else:
            final = history[-1]['result']
            return {
                'thought': f'The calculation is complete. 15 × 7 + 3 = {final}.',
                'action': 'finish',
                'params': {'answer': final},
            }

    # Fallback
    return {'thought': 'I do not know how to answer this.', 'action': 'finish', 'params': {'answer': 'unknown'}}


# ── Agent loop ────────────────────────────────────────────────────────────────

def run_agent(goal: str, tools: dict, max_steps: int = 10) -> list:
    """
    Run the ReAct agent loop. Returns the full step-by-step trace.
    """
    history = []
    print(f"Goal: {goal}")
    print("=" * 60)

    for step in range(max_steps):
        fm_output = mocked_llm(goal, history)
        action    = fm_output['action']
        params    = fm_output['params']
        thought   = fm_output['thought']

        print(f"\nStep {step + 1}")
        print(f"  Thought : {thought}")
        print(f"  Action  : {action}({params})")

        if action == 'finish':
            result = params.get('answer', '')
            print(f"  → DONE. Answer: {result}")
            history.append({'step': step + 1, 'thought': thought, 'action': action, 'params': params, 'result': result})
            break

        if action not in tools:
            result = f"Error: tool '{action}' not found"
        else:
            result = tools[action](**params)

        print(f"  Result  : {result}")
        history.append({'step': step + 1, 'thought': thought, 'action': action, 'params': params, 'result': result})

    return history


trace = run_agent("What is 15 * 7 + 3?", TOOLS)

## Hallucination as a failure mode

**Hallucination** occurs when an FM generates a confident, plausible-sounding response that is factually wrong. In an agent context this is particularly dangerous: the agent may act on the hallucinated fact.

The canonical mitigation is **tool use** — instead of asking the FM to recall facts from its weights, force it to call a tool that returns a verified result. Below we compare two scenarios:

1. **FM without tools**: asked for a calculation, it returns its parametric "memory" (which may be wrong)
2. **FM with tools**: the same question routes through the calculator, guaranteeing correctness

In practice, hallucination affects not just arithmetic (where it is obvious) but also factual claims, API signatures, file paths, and dates — wherever the FM must recall specifics rather than reason.

In [ ]:
# Simulate hallucination: FM without tools returns a confident but wrong answer

def fm_without_tools(question: str) -> str:
    """Mocked FM that answers from 'parametric memory' — sometimes wrong."""
    # Simulate a hallucination: FM confidently gives a wrong answer
    hallucinated_answers = {
        "What is 15 * 7 + 3?": "108",      # Wrong: correct is 108... wait let us use a clearly wrong example
        "What is 17 * 23?": "[FM memory] 17 × 23 = 391",   # correct is 391 — FM gets lucky
        "What is 997 * 1009?": "[FM memory] 997 × 1009 = 1,005,973",  # correct is 1,005,973
        "What is 2**32?": "[FM memory] 2^32 = 4,294,967,296",          # correct
    }
    # Deliberately wrong answers to illustrate the problem
    wrong_answers = {
        "What is 15 * 7 + 3?": "[FM memory] 15 × 7 + 3 = 110",  # Wrong! Correct = 108
        "What is 123 * 456?": "[FM memory] 123 × 456 = 56,000",  # Wrong! Correct = 56,088
    }
    if question in wrong_answers:
        return wrong_answers[question]
    return "[FM memory] I don't recall the exact figure."


def fm_with_tools(question: str, tools: dict) -> str:
    """FM that detects arithmetic questions and routes them to the calculator."""
    # Extract expression from question
    match = re.search(r'([\d\s\*\+\-\/\.\(\)]+)', question)
    if match:
        expr = match.group(1).strip()
        result = tools['calculator'](expr)
        return f"[Tool result] calculator({expr!r}) = {result}"
    return "[FM] Could not extract a calculable expression."


test_questions = [
    "What is 15 * 7 + 3?",
    "What is 123 * 456?",
]

print(f"{'Question':<30}  {'Without tools (FM memory)':<40}  {'With tools (verified)':<35}")
print("-" * 110)
for q in test_questions:
    no_tool  = fm_without_tools(q)
    with_tool = fm_with_tools(q, TOOLS)
    print(f"{q:<30}  {no_tool:<40}  {with_tool:<35}")

print("\nCorrect answers:")
print("  15 * 7 + 3 =", calculator('15 * 7 + 3'))
print("  123 * 456  =", calculator('123 * 456'))

## Visualising the agent trace

Below we plot the step-by-step trace from our ReAct agent as a timeline, showing the action taken at each step and the tool used.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

action_colors = {'calculator': BRAND, 'web_search': TEAL, 'finish': YELLOW}
x_positions   = [t['step'] for t in trace]
actions       = [t['action'] for t in trace]
colors        = [action_colors.get(a, ROSE) for a in actions]
labels        = [f"{t['action']}\n{t['result'][:20]}..." if len(str(t['result'])) > 20
                 else f"{t['action']}\n{t['result']}" for t in trace]

for i, (x, label, color) in enumerate(zip(x_positions, labels, colors)):
    ax.scatter(x, 0.5, s=300, color=color, zorder=3)
    ax.text(x, 0.65, label, ha='center', va='bottom', fontsize=9, color='#e2e8f0')
    if i > 0:
        ax.annotate('', xy=(x - 0.05, 0.5), xytext=(x_positions[i-1] + 0.05, 0.5),
                    arrowprops=dict(arrowstyle='->', color=MUTED, lw=1.5))

legend_patches = [mpatches.Patch(color=c, label=a) for a, c in action_colors.items()]
ax.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=9)

ax.set_xlim(0.5, len(trace) + 0.5)
ax.set_ylim(0, 1.4)
ax.set_xticks(x_positions)
ax.set_xticklabels([f'Step {x}' for x in x_positions])
ax.set_yticks([])
ax.set_title('ReAct agent trace: goal → perception → reasoning → action → result')
plt.tight_layout()
plt.show()

## ✏️ Your turn

**Exercise — Add a clock tool to the agent.**

The agent currently has `calculator` and `web_search`. Your task:

1. Implement a `clock()` tool that returns the current time as a string (e.g. `"2026-06-18 14:30:00"`)
2. Add it to the `TOOLS` registry
3. Update `mocked_llm` to route time-related questions to the clock tool
4. Run the agent on `"What time is it?"` and assert it calls the clock tool at least once

In [ ]:
from datetime import datetime

# TODO(you): implement a clock tool
def clock() -> str:
    """Return the current local time as a string."""
    pass  # replace with your implementation


# TODO(you): add clock to the tools registry
TOOLS_EXTENDED = dict(TOOLS)  # copy existing tools
# TOOLS_EXTENDED['clock'] = clock


# TODO(you): update mocked_llm or create a new version that routes
# "What time is it?" to the clock tool
def mocked_llm_v2(goal: str, history: list) -> dict:
    if 'time' in goal.lower():
        if len(history) == 0:
            return {
                'thought': 'The user wants to know the current time. I will use the clock tool.',
                'action': 'clock',
                'params': {},
            }
        else:
            result = history[-1]['result']
            return {
                'thought': f'The clock returned: {result}.',
                'action': 'finish',
                'params': {'answer': result},
            }
    # Delegate to original mocked_llm for other goals
    return mocked_llm(goal, history)


# TODO(you): run the agent with the new tool registry and LLM version
# trace_clock = run_agent_v2("What time is it?", TOOLS_EXTENDED)
trace_clock = None  # replace with a real run

In [ ]:
# Assertion cell — runs silently when correct
assert clock is not None, "Implement the clock() function"
assert clock() is not None, "clock() must return a non-None value"

# Check that the clock tool returns something that looks like a time string
clock_result = clock()
assert isinstance(clock_result, str) and len(clock_result) > 0, \
    "clock() should return a non-empty string"

assert 'clock' in TOOLS_EXTENDED, "Add 'clock' to TOOLS_EXTENDED"

assert trace_clock is not None, "Run the agent on 'What time is it?' and assign the trace"

clock_used = any(step['action'] == 'clock' for step in trace_clock)
assert clock_used, "The agent should use the clock tool when asked 'What time is it?'"

print("✓ Exercise passed — agent correctly uses the clock tool")

<details>
<summary>Solution</summary>

```python
from datetime import datetime

def clock() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

TOOLS_EXTENDED = dict(TOOLS)
TOOLS_EXTENDED['clock'] = clock

# Replace run_agent's mocked_llm call with mocked_llm_v2:
def run_agent_v2(goal, tools, max_steps=10):
    history = []
    print(f"Goal: {goal}")
    print("=" * 60)
    for step in range(max_steps):
        fm_output = mocked_llm_v2(goal, history)
        action = fm_output['action']
        params = fm_output['params']
        thought = fm_output['thought']
        print(f"\nStep {step + 1}")
        print(f"  Thought : {thought}")
        print(f"  Action  : {action}({params})")
        if action == 'finish':
            result = params.get('answer', '')
            print(f"  → DONE. Answer: {result}")
            history.append({'step': step+1, 'thought': thought, 'action': action, 'params': params, 'result': result})
            break
        result = tools[action](**params) if action in tools else f"Error: tool '{action}' not found"
        print(f"  Result  : {result}")
        history.append({'step': step+1, 'thought': thought, 'action': action, 'params': params, 'result': result})
    return history

trace_clock = run_agent_v2("What time is it?", TOOLS_EXTENDED)
```
</details>